In [ ]:
import os
import json
import pandas as pd


In [ ]:
# Path to folder containing your .txt files
folder_path = r"Dataset"  # <-- Change this to your actual folder


# Data collection
merged_mcq_data = []  # for JSON dump
df_rows = []          # for DataFrame

In [ ]:
# Traverse subfolders
for split in ['train', 'dev', 'test']:
    split_folder = os.path.join(folder_path, split)

    for filename in os.listdir(split_folder):
        if filename.endswith('.txt'):
            file_path = os.path.join(split_folder, filename)

            with open(file_path, 'r', encoding='utf-8') as f:
                item = json.load(f)

            article = item.get("article", "")
            questions = item.get("questions", [])
            answers = item.get("answers", [])
            options = item.get("options", [])

            mcqs = []
            for q, a, opt in zip(questions, answers, options):
                mcqs.append({
                    "question": q,
                    "answer": a,
                    "options": opt
                })

            # Append to both datasets
            merged_mcq_data.append({
                "article": article,
                "questions": questions,
                "answers": answers,
                "options": options
            })

            df_rows.append({
                "Article": article,
                "MCQs": mcqs,
                "MCQ Count": len(mcqs)
            })


In [ ]:
# Create DataFrame
df = pd.DataFrame(df_rows)

# Save JSON backup of full dataset
output_path = os.path.join(folder_path, "merged_dataset_backup.json")
df.to_json(output_path, orient="records", indent=2, force_ascii=False)

print(f"✅ Dataset loaded with {len(df)} articles.")
print(f"📦 Backup JSON saved to: {output_path}")

In [ ]:
df.head()

### Formatting Dataset

In [ ]:
# dev.json, train.json, test.json
import pandas as pd
import os

file_name = "train.json"

df = pd.read_json(rf"Dataset\Semi-Format\\{file_name}")

## Creating Prompt Structure

In [16]:
import random

def generate_mcq_prompts(mcq_count):
    templates = [
        f"Generate {mcq_count} MCQs from the following context.",
        f"Create {mcq_count} multiple choice questions based on the given context.",
        f"Write {mcq_count} MCQs using the information provided below.",
        f"From the context provided, generate {mcq_count} MCQs.",
        f"Produce {mcq_count} high-quality MCQs from this content.",
        f"Based on the following material, generate {mcq_count} multiple choice questions.",
        f"Generate {mcq_count} concept-based MCQs from the given passage.",
        f"Using the provided context, write {mcq_count} MCQs.",
        f"As an AI tutor, generate {mcq_count} MCQs from the following text.",
        f"Make {mcq_count} practice MCQs based on this context.",
        f"Create {mcq_count} entry-test style MCQs from the content below.",
]
    return random.sample(templates,k=1)[0]  # get 5 random variations
df.head()

,Article,MCQs,MCQ Count
0,Studies show that you may be lied to every day...,[{'question': 'From Para.1 we learn that lying...,5
1,"You could feel sorry for Alberto Torres, who i...",[{'question': 'Mr. Torres became blind when _...,6
2,Although most weddings follow long-held tradit...,[{'question': 'Which of the following best sho...,3
3,It was 1961 and I was in the fifth grade. My m...,[{'question': 'We can learn from the beginning...,4
4,The SAT is one of two major tests for the entr...,[{'question': 'What will be tested in the new ...,3


In [17]:
df ["query"] = df["MCQ Count"].apply(generate_mcq_prompts)
df ["query"] = df ["query"] + "\n\n" + df["Article"]

df.drop(["Article", "MCQ Count"], axis=1,inplace=True)
df.rename(columns={"MCQs": "response"}, inplace=True)

df.head()


,response,query
0,[{'question': 'From Para.1 we learn that lying...,Write 5 MCQs using the information provided be...
1,[{'question': 'Mr. Torres became blind when _...,"Based on the following material, generate 6 mu..."
2,[{'question': 'Which of the following best sho...,Write 3 MCQs using the information provided be...
3,[{'question': 'We can learn from the beginning...,Write 4 MCQs using the information provided be...
4,[{'question': 'What will be tested in the new ...,"Using the provided context, write 3 MCQs.\n\nT..."


In [ ]:
# Input and output file paths
folder_path = r"Dataset\Final Format"
output_path = os.path.join(folder_path, file_name)

# Flatten 'response' column by converting nested objects (like dicts/lists) to string
def convert_to_str(example):
    formatted_string = ""
    for idx, item in enumerate(example, start=1):
        formatted_string += f"{idx}) {item['question']}\n"
        for opt_label, option in zip("abcd", item['options']):
            formatted_string += f"   {opt_label}) {option}\n"
        formatted_string += f"Correct Answer: {item['answer'].lower()}\n\n"

    # Print or store this string
    return formatted_string

df["response"] = df["response"].apply(convert_to_str)

In [19]:

# Save the cleaned DataFrame back to JSON
df.to_json(output_path, orient="records", indent=2, force_ascii=False)


In [ ]:
# dev.json, train.json, test.json
import pandas as pd
import os

file_name = "test.json"

df = pd.read_json(rf"Dataset\Final Format\{file_name}")

In [42]:
len(max(df["response"]))

931

## Now the dataset is in the ready state and we will proceed with the training